# Exploring London’s Public Transport Usage with Snowflake SQL

London—historically known as *Londinium*—is one of the world’s most diverse and densely populated cities, with more than 8.5 million residents speaking over 300 languages. While the original “Square Mile” remains a small historic core, modern Greater London now spans 32 boroughs across more than 600 square miles.

As the city expanded, so did the demand for an efficient transportation system. Today, **Transport for London (TfL)** manages London’s integrated network of buses, trams, the Underground, Overground, DLR, river services, taxis, and even the Emirates Airline cable car. Journey data from these services provides a valuable look at how people move across the city over time.

For this analysis, I worked with a curated version of TfL’s publicly available journey dataset, stored in a **Snowflake** database named `TFL`. The table `JOURNEYS` contains monthly journey counts, broken down by transport type and measured in millions.

## Dataset Overview: `TFL.JOURNEYS`

The table includes one row per month, year, and transport category.

| Column | Description | Type |
|--------|-------------|------|
| `MONTH` | Month number (1 = January) | INTEGER |
| `YEAR` | Calendar year | INTEGER |
| `DAYS` | Number of days in the month | INTEGER |
| `REPORT_DATE` | Date the record was reported | DATE |
| `JOURNEY_TYPE` | Transport mode (e.g., Underground & DLR, Buses, Emirates Airline) | VARCHAR |
| `JOURNEYS_MILLIONS` | Monthly journey volume in millions | FLOAT |

Snowflake stores object names in uppercase by default, so queries refer to the table as `TFL.JOURNEYS`.

## Analysis Objectives

To understand Londoners’ travel patterns, I explored three key questions:

1. **Which transport types have the highest total journey volumes overall?**
2. **For the Emirates Airline cable car, which month–year combinations saw the highest ridership?**
3. **For Underground & DLR services, which years recorded the lowest usage?**

The SQL queries below were executed against the Snowflake table, and the resulting outputs were stored in the following DataFrames:

- `most_popular_transport_types`
- `emirates_airline_popularity`
- `least_popular_years_tube`


## 1. Most Popular Transport Modes

To compare overall usage, I aggregated journey volumes across the entire timeframe for each transport mode.

```sql
-- most_popular_transport_types
SELECT
    JOURNEY_TYPE,
    SUM(JOURNEYS_MILLIONS) AS TOTAL_JOURNEYS_MILLIONS
FROM TFL.JOURNEYS
GROUP BY JOURNEY_TYPE
ORDER BY TOTAL_JOURNEYS_MILLIONS DESC;
```

This query sums journey volumes for each transport type and ranks the results from highest to lowest.


## 2. Highest-Ridership Months for the Emirates Airline

The Emirates Airline is one of London’s more unique transport modes, connecting Greenwich Peninsula and the Royal Docks by cable car. To find when it was most popular, I identified the five month–year combinations with the highest journey volumes.

```sql
-- emirates_airline_popularity
SELECT
    MONTH,
    YEAR,
    ROUND(JOURNEYS_MILLIONS, 2) AS ROUNDED_JOURNEYS_MILLIONS
FROM TFL.JOURNEYS
WHERE
    JOURNEY_TYPE = 'Emirates Airline'
    AND JOURNEYS_MILLIONS IS NOT NULL
ORDER BY
    ROUNDED_JOURNEYS_MILLIONS DESC
LIMIT 5;
```

This query returns the top five months with the highest cable car usage.


## 3. Least Popular Years for Underground & DLR

Finally, I examined the Underground & DLR category to identify the years with the lowest overall journey volumes.

```sql
-- least_popular_years_tube
SELECT
    YEAR,
    JOURNEY_TYPE,
    SUM(JOURNEYS_MILLIONS) AS TOTAL_JOURNEYS_MILLIONS
FROM TFL.JOURNEYS
WHERE JOURNEY_TYPE = 'Underground & DLR'
GROUP BY YEAR, JOURNEY_TYPE
ORDER BY TOTAL_JOURNEYS_MILLIONS
LIMIT 5;
```

This reveals the five lowest-volume years for the Underground & DLR network.
